# Step 1: Build Target Labels

Build binary outlier labels from response_outliers.csv

## Imports

In [1]:
import time
from pathlib import Path
import pandas as pd

In [2]:
# AUTO-DETECT PROJECT ROOT
import os
from pathlib import Path

def find_project_root(marker_file='CLAUDE.md', max_depth=5):
    """Find project root by looking for marker file"""
    current = Path.cwd()
    
    # Try current directory and parents
    for _ in range(max_depth):
        if (current / marker_file).exists():
            return current
        current = current.parent
    
    # If not found, return current working directory
    print(f"WARNING: Could not find {marker_file}, using current directory")
    return Path.cwd()

# Find and change to project root
project_root = find_project_root()
os.chdir(project_root)

print(f"Project root: {project_root}")
print(f"Current directory: {Path.cwd()}")
print(f"\nData directory exists: {(Path.cwd() / 'data').exists()}")
print(f"Outputs directory exists: {(Path.cwd() / 'outputs').exists()}")

Project root: d:\capstone_pipeline
Current directory: d:\capstone_pipeline

Data directory exists: True
Outputs directory exists: True


## Configuration

In [3]:
# Configuration parameters
data_dir = "data"
output_dir = "outputs"
force = False  # Set to True to force rerun

## Setup and Initialization

In [4]:
start_time = time.time()
print("=" * 60)
print("Step 1: Building Target Labels")
print("=" * 60)

# Setup paths
data_dir = Path(data_dir)
output_dir = Path(output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
features_dir = output_dir / "features"
features_dir.mkdir(parents=True, exist_ok=True)

sentinel = features_dir / "01_target.done"
if sentinel.exists() and not force:
    print(f"[OK] Target already built (found {sentinel})")
    print(f"  Use --force to rebuild")
else:
    # Check input files
    outliers_file = data_dir / "response_outliers.csv"
    response_file = data_dir / "response_updated.csv"

    if not outliers_file.exists():
        raise FileNotFoundError(
            f"Could not find {outliers_file}\n"
            f"Please ensure response_outliers.csv is in the data/ directory.\n"
            f"See DATA SETUP section in CLAUDE.md"
        )

    if not response_file.exists():
        raise FileNotFoundError(
            f"Could not find {response_file}\n"
            f"Please ensure response_updated.csv is in the data/ directory.\n"
            f"See DATA SETUP section in CLAUDE.md"
        )

Step 1: Building Target Labels
[OK] Target already built (found outputs\features\01_target.done)
  Use --force to rebuild


## Load Data

In [5]:
# Load outlier wafers
print(f"Loading outlier list from {outliers_file}...")
outliers_df = pd.read_csv(outliers_file)
print(f"  Found {len(outliers_df)} outlier wafers")

# Load response data to get PARAM_END_DATETIME and LOT_ID
print(f"Loading response data from {response_file}...")
response_df = pd.read_csv(response_file)
print(f"  Loaded {len(response_df)} rows")

NameError: name 'outliers_file' is not defined

## Process and Create Target

In [ ]:
# Get unique wafers with their LOT_ID and PARAM_END_DATETIME
# Take the first PARAM_END_DATETIME for each wafer (they should be the same)
wafer_info = response_df[['WAFER_SCRIBE', 'LOT_ID', 'PARAM_END_DATETIME']].drop_duplicates('WAFER_SCRIBE')
print(f"  Found {len(wafer_info)} unique wafers in response data")

# Mark outliers
wafer_info['is_outlier'] = wafer_info['WAFER_SCRIBE'].isin(outliers_df['WAFER_SCRIBE']).astype(int)

## Save Results

In [ ]:
# Save target
output_file = features_dir / "target.parquet"
print(f"Saving target to {output_file}...")
wafer_info.to_parquet(output_file, index=False)

# Print statistics
print("\n" + "=" * 60)
print("Target Statistics:")
print("=" * 60)
print(f"Total wafers: {len(wafer_info)}")
print(f"Outliers: {wafer_info['is_outlier'].sum()} ({100 * wafer_info['is_outlier'].mean():.2f}%)")
print(f"Non-outliers: {(1 - wafer_info['is_outlier']).sum()} ({100 * (1 - wafer_info['is_outlier']).mean():.2f}%)")
print(f"\nClass balance: {wafer_info['is_outlier'].value_counts().to_dict()}")

# Mark complete
sentinel.touch()

elapsed = time.time() - start_time
print(f"\n[OK] Target build complete in {elapsed:.2f}s")
print("=" * 60)